# Inference & Feature Explorer — BioS birthday recall

A focused playground for the trained `grid-L4-H6` CLT model. Four sections:

1. **Classic Inference** — write a prompt template (`{name} was born on`), fill it from any person by index, and see what the model predicts. End the template wherever you like to probe a different field (month → day → year).
2. **Specified Inference** — same, but pick the person by **filtering on birth month** and indexing into that subset (it reports each person's absolute dataset index).
3. **Feature Graph** — build a circuit-tracer attribution graph for the current prompt (top feature nodes + interactive web viewer), the same flow as `explore_attribution.ipynb`.
4. **Feature Analysis** — sweep the CLT features that fire when the model recalls a birth month across many people (mean activation + specificity).

> Kernel must be the `clts/.venv-ct` interpreter. All boilerplate (loading, helpers) lives in **hidden cells** — click to expand if you want to read it. Run the loading cells once (top to bottom), then jump to whichever section you want.

## Loading  *(hidden — run once, top to bottom)*

In [59]:
import re
import sys
import time
from collections import defaultdict
from pathlib import Path

import torch

# Notebook lives in clts/; run everything relative to the repo root.
REPO = Path.cwd()
if REPO.name == "clts":
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

# Fixed inputs — only one trained CLT exists.
MODEL_DIR = REPO / "model/grid-L4-H6"
CLT_DIR   = REPO / "clts/clt_runs/grid-L4-H6/mult16_l02_lr0.0001_ep50_n10000/final"
DATA_DIR  = REPO / "data/bioS_N-Bd_final_grid"
SCAN_NAME = "grid-L4-H6"
DEVICE    = "cpu"

from clts.storage import storage_root
from util.bio_sampler import BioSampler
from util.condensed_tokenizer import CondensedTokenizer

# Tokenizer is used only for the in-vocab name check (skip people whose name has
# out-of-vocab subwords — the model can't cleanly recall those).
ct = CondensedTokenizer.from_remap_path(DATA_DIR / "old_to_new.json")
sampler = BioSampler(DATA_DIR / "people.json", fields=("birthday",))

def name_in_vocab(name: str) -> bool:
    try:
        ct.encode(name)
        return True
    except KeyError:
        return False

print(f"{len(sampler.people):,} people loaded · repo={REPO.name}")

50,000 people loaded · repo=Interp_LM4


In [60]:
from clts.export_tokenizer import ensure_hf_tokenizer
from clts.load_replacement_model import load_replacement_model

# Load the circuit-tracer replacement model (custom Llama + trained CLT) ONCE
# and cache it. Set `model = None` and re-run to force a reload. First load
# takes ~10 s on CPU.
if globals().get("model") is None:
    model = load_replacement_model(
        MODEL_DIR, CLT_DIR, ensure_hf_tokenizer(DATA_DIR), SCAN_NAME, device=DEVICE
    )
    print(f"replacement model loaded — n_layers={model.cfg.n_layers}, "
          f"d_model={model.cfg.d_model}, d_vocab={model.cfg.d_vocab}")
else:
    print("model already loaded (set `model = None` and re-run to reload)")

model already loaded (set `model = None` and re-run to reload)


In [61]:
# ===== prompt building =======================================================
# Fill a template string from a person dict. The "middle text" (e.g. " was born
# on ") is just literal text in the template — end the template right before the
# field you want the model to PREDICT.
#
# Placeholders:
#   {name} {first} {middle} {last} {month} {day} {year} {date}
#   {city} {university} {field} {company} {company_city} {he_she}
#
# A leading space is prepended automatically (the model saw every bio starting
# " <Name> ..."), so prompts stay on the exact training distribution.

def person_fields(person: dict) -> dict:
    return {
        "name":         f"{person['first_name']} {person['middle_name']} {person['last_name']}",
        "first":        person["first_name"],
        "middle":       person["middle_name"],
        "last":         person["last_name"],
        "month":        person["birthmonth"],
        "day":          person["birthday"],
        "year":         person["birthyear"],
        "date":         f"{person['birthmonth']} {person['birthday']}, {person['birthyear']}",
        "city":         person.get("birthcity", ""),
        "university":   person.get("university", ""),
        "field":        person.get("field", ""),
        "company":      person.get("company1name", ""),
        "company_city": person.get("company1city", ""),
        "he_she":       "He" if person["id"] % 2 == 0 else "She",
    }

def build_prompt(template: str, person: dict | None = None) -> str:
    """Fill `template` from `person` (or pass it through verbatim if person is
    None / has no placeholders). Guarantees a single leading space."""
    text = template.format_map(person_fields(person)) if person is not None else template
    if text and not text[0].isspace():
        text = " " + text
    return text

# ===== inference =============================================================

@torch.no_grad()
def predict(prompt, expected=None, top_k=10, n_continue=0):
    """Run the model on `prompt`; print the top-k next tokens and (optionally) a
    greedy continuation that "fills in the blank". `expected` is a token string
    (e.g. ' August') to mark and rank. Returns the next-token probability vector."""
    tokens = model.ensure_tokenized(prompt)
    logits = model(tokens.unsqueeze(0))
    probs = logits[0, -1].softmax(-1)

    exp_id = None
    if expected is not None:
        ids = model.tokenizer.encode(expected, add_special_tokens=False)
        exp_id = ids[0] if len(ids) == 1 else None

    print(f"prompt: {prompt!r}")
    top_p, top_i = probs.topk(top_k)
    print(f"\ntop {top_k} next tokens:")
    for rank, (pi, pp) in enumerate(zip(top_i.tolist(), top_p.tolist()), 1):
        mark = "  <- expected" if (exp_id is not None and pi == exp_id) else ""
        print(f"  {rank:>2}. {model.tokenizer.decode([pi])!r:<14} p={pp:.3f}{mark}")

    if expected is not None and exp_id is not None:
        rank = int((probs > probs[exp_id]).sum()) + 1
        print(f"\nexpected {expected!r}: p={probs[exp_id]:.3f}, rank {rank}/{probs.numel()}")
    elif expected is not None:
        print(f"\nexpected {expected!r} is not a single token; not ranked.")

    if n_continue > 0:
        out = tokens.clone()
        for _ in range(n_continue):
            step = model(out.unsqueeze(0))
            out = torch.cat([out, step[0, -1].argmax()[None]])
        cont = model.tokenizer.decode(out[len(tokens):].tolist())
        print(f"\ngreedy +{n_continue} tok: {prompt!r} -> {prompt + cont!r}")
    return probs

# ===== person selection ======================================================

def pick_person(idx, in_vocab_only=True):
    """The idx-th person (0-based). With in_vocab_only, count only people whose
    name tokenizes cleanly. Returns (person, dataset_idx) where dataset_idx is
    the absolute position in sampler.people."""
    count = 0
    for ds_idx, p in enumerate(sampler.people):
        if in_vocab_only and not name_in_vocab(f" {p['first_name']} {p['last_name']}"):
            continue
        if count == idx:
            return p, ds_idx
        count += 1
    raise IndexError(f"idx={idx} past the last "
                     f"{'in-vocab ' if in_vocab_only else ''}person")

def people_in_month(month, in_vocab_only=True):
    """All (dataset_idx, person) born in `month`, in dataset order."""
    out = []
    for ds_idx, p in enumerate(sampler.people):
        if p["birthmonth"] != month:
            continue
        if in_vocab_only and not name_in_vocab(f" {p['first_name']} {p['last_name']}"):
            continue
        out.append((ds_idx, p))
    return out

# ===== expected-token resolution ============================================

def expected_next_token(prompt, person, exposure_idx=0):
    """Ground-truth next token after `prompt`, IF `prompt` is a clean prefix of
    the person's rendered bio (template `exposure_idx`). Returns a token string
    like ' August', or None if the prompt diverges (e.g. custom middle text).
    This is what lets PREDICT='auto' track the field as you slide the cut point."""
    full = sampler.render(person, exposure_idx)
    if not (full.startswith(prompt) and len(full) > len(prompt)):
        return None
    full_ids = model.tokenizer.encode(full, add_special_tokens=False)
    prompt_ids = model.tokenizer.encode(prompt, add_special_tokens=False)
    if full_ids[:len(prompt_ids)] == prompt_ids and len(full_ids) > len(prompt_ids):
        return model.tokenizer.decode([full_ids[len(prompt_ids)]])
    return None

def resolve_expected(predict, prompt, person):
    """Map the PREDICT knob to a token string to rank in predict().
      'auto'              -> infer from the ground-truth bio (default)
      'month'/'day'/'year'-> that field, as a leading-space token
      None                -> don't rank anything
      any other string    -> used literally as the expected token"""
    if predict is None:
        return None
    if predict == "auto":
        return expected_next_token(prompt, person)
    field = {"month": "birthmonth", "day": "birthday", "year": "birthyear"}.get(predict)
    if field is not None:
        return " " + str(person[field])
    return predict   # treat as a literal token string

print("helpers ready: build_prompt, predict, pick_person, people_in_month, resolve_expected")

helpers ready: build_prompt, predict, pick_person, people_in_month, resolve_expected


## 1 · Classic Inference

Pick a person by index, write a `TEMPLATE`, and see what the model predicts. **End the template right before the field you want predicted** — the middle text is yours to edit:

| `TEMPLATE` | model predicts |
|---|---|
| `"{name} was born on"` | the **month** |
| `"{name} was born on {month}"` | the **day** |
| `"{name} was born on {month} {day},"` | the **year** |
| `"{name} studied at"` | their **university** |

Placeholders: `{name} {first} {middle} {last} {month} {day} {year} {date} {city} {university} {field} {company} {company_city} {he_she}`. You can also write a totally free-form string (no placeholders).

`PREDICT` controls which token gets its probability/rank reported: leave it `"auto"` and it figures out the ground-truth next token from the bio (so just sliding the cut point from month → day → year Just Works), or pin it to `"month"`/`"day"`/`"year"`/`None`. `N_CONTINUE` greedily fills in that many tokens after the prompt.

In [62]:
PERSON_IDX = 0                      # 0, 1, 2, ... over in-vocab people
TEMPLATE   = "{name} was born on the memorable date of"   # edit the middle text freely; end before the field to predict
PREDICT    = "auto"                 # "auto" | "month" | "day" | "year" | None  (token to rank)
N_CONTINUE = 4                      # greedily fill in this many tokens after the prompt

person, ds_idx = pick_person(PERSON_IDX)
print(f"person #{PERSON_IDX}  (dataset index {ds_idx}, id {person['id']})")
print(f"  full name : {person['first_name']} {person['middle_name']} {person['last_name']}")
print(f"  birthday  : {person['birthmonth']} {person['birthday']}, {person['birthyear']}")
print()

prompt = build_prompt(TEMPLATE, person)
predict(prompt, expected=resolve_expected(PREDICT, prompt, person), n_continue=N_CONTINUE)

person #0  (dataset index 11, id 22)
  full name : Gage Wyatt Clay
  birthday  : August 24, 1712

prompt: ' Gage Wyatt Clay was born on the memorable date of'

top 10 next tokens:
   1. ' August'      p=1.000
   2. ' December'    p=0.000
   3. ' May'         p=0.000
   4. ' January'     p=0.000
   5. 'lyn'          p=0.000
   6. ' November'    p=0.000
   7. ' October'     p=0.000
   8. 'lin'          p=0.000
   9. ' June'        p=0.000
  10. ' Leonardo'    p=0.000

greedy +4 tok: ' Gage Wyatt Clay was born on the memorable date of' -> ' Gage Wyatt Clay was born on the memorable date of August 24, 17'


tensor([1.8779e-15, 4.8978e-19, 3.1025e-17,  ..., 9.0225e-19, 1.8077e-18,
        6.5065e-16])

## 2 · Specified Inference  *(filter by birth month)*

Same inference, but choose the person by **filtering on birth month** and indexing into that subset. It lists the first several people in the subset with their **absolute dataset index** and `id`, so you can cross-reference them elsewhere, then runs inference on the one you select. The `prompt` it builds is also what section 3 (Feature Graph) will trace.

In [161]:
MONTH      = "December"               # filter people born in this month
SUBSET_IDX =  1# 0, 1, 2, ... WITHIN the month subset
TEMPLATE   = "{name} was born on the memorable date of"   # same template knobs as section 1
PREDICT    = "auto"                 # "auto" | "month" | "day" | "year" | None
N_CONTINUE = 4

subset = people_in_month(MONTH)     # [(dataset_idx, person), ...], in-vocab only
print(f"{len(subset)} in-vocab people born in {MONTH}.  First few:")
for sub_i, (d_idx, p) in enumerate(subset[:8]):
    print(f"  subset #{sub_i:<3} -> dataset index {d_idx:<6} id {p['id']:<6} "
          f"{p['first_name']} {p['last_name']}")

ds_idx, person = subset[SUBSET_IDX]
print(f"\nselected subset #{SUBSET_IDX}:  dataset index {ds_idx},  id {person['id']}")
print(f"  full name : {person['first_name']} {person['middle_name']} {person['last_name']}")
print(f"  birthday  : {person['birthmonth']} {person['birthday']}, {person['birthyear']}")
print()

prompt = build_prompt(TEMPLATE, person)
predict(prompt, expected=resolve_expected(PREDICT, prompt, person), n_continue=N_CONTINUE)

132 in-vocab people born in December.  First few:
  subset #0   -> dataset index 328    id 657    Kayla Croft
  subset #1   -> dataset index 411    id 822    Gage Joseph
  subset #2   -> dataset index 655    id 1311   Gianna Bowles
  subset #3   -> dataset index 1704   id 3408   Patrick Buck
  subset #4   -> dataset index 2134   id 4269   Kayla Goodall
  subset #5   -> dataset index 2798   id 5597   Kylie Mead
  subset #6   -> dataset index 2848   id 5697   Gianna Lawton
  subset #7   -> dataset index 2867   id 5734   Patrick Islam

selected subset #1:  dataset index 411,  id 822
  full name : Gage Camden Joseph
  birthday  : December 20, 1773

prompt: ' Gage Camden Joseph was born on the memorable date of'

top 10 next tokens:
   1. ' December'    p=1.000
   2. ' April'       p=0.000
   3. ' May'         p=0.000
   4. ' July'        p=0.000
   5. ' March'       p=0.000
   6. ' October'     p=0.000
   7. ' January'     p=0.000
   8. ' 8'           p=0.000
   9. ' 18'          p=0.000
 

tensor([4.8385e-20, 1.1970e-23, 1.0950e-20,  ..., 7.7022e-20, 4.9964e-21,
        3.5599e-20])

In [162]:
GRAPH_TARGET = None   # None -> auto (expected token for `prompt`, else top prediction); or e.g. " August"

feature_graph(prompt, target=GRAPH_TARGET)


prompt : ' Gage Camden Joseph was born on the memorable date of'
target : ' December'
slug   : gage-camden-joseph-was-born-on-the-memorable-date-of

[tokenizer] cache hit: /Users/efmac/Code/Project Code/CRL-Interp/Interp_LM4/clt_storage/hf_tokenizers/0518e632
Moving model to device:  cpu
🆕 NEW graph 'gage-camden-joseph-was-born-on-the-memorable-date-of'.
wrote gage-camden-joseph-was-born-on-the-memorable-date-of.json → /Users/efmac/Code/Project Code/CRL-Interp/Interp_LM4/clt_storage/clt_graphs/grid-L4-H6/explore


{'prompt': ' Gage Camden Joseph was born on the memorable date of',
 'scan_name': 'grid-L4-H6',
 'top_logit_token': ' December',
 'target_logit_prob': 1.0,
 'replacement_score': 0.5358524918556213,
 'completeness_score': 0.876301646232605,
 'error_influence_share': 0.46414750814437866,
 'n_feature_nodes_after_pruning': 276}

## 3 · Feature Graph — trace the recall circuit

One call builds a circuit-tracer attribution graph for the **current `prompt`** (set by section 1 or 2) toward a target token. Everything else lives in the hidden cell below — click to expand if you want to read it.

```python
feature_graph(prompt)     # build the graph (prints status, returns the report)
show_top_features()       # 3a · inline top feature nodes
serve_viewer()            # 3b · interactive web viewer (run once)
```

Graphs are written to the shared `clt_graphs/<scan>/explore/` dir, so they appear in the same viewer dropdown as `explore_attribution.ipynb`. Run section 1 or 2 first to set `prompt`. `GRAPH_TARGET = None` auto-picks the target (the expected token for the prompt, else the model's top prediction); set it to a token string like `" August"` to trace a specific one.

In [63]:
# ===== Section 3 helpers  (hidden — defines the one-call API) ================
# Three calls, one per subsection. They share the most-recent build via `_LAST`:
#   feature_graph(prompt[, target])  -> build the attribution graph        (3)
#   show_top_features([top_k])       -> inline top-feature bar chart        (3a)
#   serve_viewer([port])             -> interactive web viewer              (3b)

import importlib
import re

import matplotlib.pyplot as plt
from circuit_tracer.graph import compute_node_influence

import clts.build_attribution_graph as _bag
from clts.serve_ui import start_server

_LAST = {}   # most recent build: prompt, target, slug, graph, report, graph_dir, server


def feature_graph(prompt, target=None, *, reload=True, verbose=False):
    """Build the circuit-tracer attribution graph for `prompt` toward `target`.

    target=None auto-picks the target: the expected (ground-truth) token for
    `prompt` if a `person` is in scope, else the model's top prediction. Pass a
    token string like " August" to trace a specific one. Prints status, stashes
    everything in `_LAST` for show_top_features()/serve_viewer(), and returns the
    report (so the calling cell displays it).

    Graphs are written to the shared clt_graphs/<scan>/explore/ dir, so they show
    up in the same viewer dropdown as explore_attribution.ipynb.
    """
    if reload:
        importlib.reload(_bag)   # pick up edits to build_attribution_graph.py

    # Resolve the attribution target.
    if target is None:
        person = globals().get("person")
        auto = resolve_expected("auto", prompt, person) if person is not None else None
        if auto is not None:
            target = auto
        else:
            with torch.inference_mode():
                _t = model.ensure_tokenized(prompt)
                target = model.tokenizer.decode([int(model(_t.unsqueeze(0))[0, -1].argmax())])

    slug = re.sub(r"[^a-z0-9]+", "-", prompt.lower()).strip("-")[:60] or "graph"
    graph_dir = storage_root() / "clt_graphs" / SCAN_NAME / "explore"

    print(f"prompt : {prompt!r}")
    print(f"target : {target!r}")
    print(f"slug   : {slug}\n")

    # build_graph reloads the model from disk internally (~10 s), independent of
    # the cached inference `model` above — same as explore_attribution.ipynb.
    result = _bag.build_graph(
        model_dir=MODEL_DIR, clt_dir=CLT_DIR, data_dir=DATA_DIR,
        scan_name=SCAN_NAME, graph_dir=graph_dir, slug=slug,
        prompt=prompt, target=target, device=DEVICE, verbose=verbose,
    )

    # Reopening an arranged graph, or starting fresh?
    status = result.get("status", "rebuilt")
    lc = result.get("layout_counts", {"pins": 0, "supernodes": 0, "renames": 0})
    if status == "new":
        print(f"🆕 NEW graph '{slug}'.")
    elif status == "reused":
        print(f"📂 REOPENED '{slug}' — layout restored "
              f"({lc['pins']} pins, {lc['supernodes']} groups, {lc['renames']} renames).")
    elif status == "inputs-changed":
        print(f"⚠️  '{slug}' inputs changed — fresh graph (old layout backed up).")
    else:
        print(f"🔁 Rebuilt '{slug}'.")
    print(f"wrote {slug}.json → {graph_dir}")

    _LAST.update(prompt=prompt, target=target, slug=slug,
                 graph=result["graph"], report=result["report"], graph_dir=graph_dir)
    return result["report"]


def show_top_features(top_k=15):
    """3a · Inline bar chart of the top feature nodes by influence on the target
    logit (circuit-tracer's compute_node_influence), for the latest build."""
    if "graph" not in _LAST:
        raise NameError("No graph yet — run feature_graph(prompt) first.")
    graph, report = _LAST["graph"], _LAST["report"]

    A = graph.adjacency_matrix.cpu()
    n_logits = len(graph.logit_targets)
    n_features = len(graph.selected_features)

    # logit_weights: weight only the logit nodes (last n_logits), as prune_graph does.
    logit_weights = torch.zeros(A.shape[0])
    logit_weights[-n_logits:] = graph.logit_probabilities.cpu()
    node_influence = compute_node_influence(A, logit_weights)

    feat_infl = node_influence[:n_features]
    order = torch.argsort(feat_infl, descending=True)[:top_k]

    labels, vals = [], []
    for i in order.tolist():
        layer, pos, fidx = graph.active_features[graph.selected_features[i]].tolist()
        act = graph.activation_values[i].item()
        labels.append(f"L{layer} F{fidx} @pos{pos}  (act {act:.2f})")
        vals.append(feat_infl[i].item())

    fig, ax = plt.subplots(figsize=(8, 0.42 * len(labels) + 1.8))
    ax.barh(range(len(labels)), vals, color="#3b7dd8")
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels)
    ax.invert_yaxis()
    ax.set_xlabel("node influence on target logit")
    ax.set_title(
        f"Top {len(labels)} feature nodes — {report['prompt']!r}\n"
        f"target {report['top_logit_token']!r}  p={report['target_logit_prob']:.2f}   |   "
        f"replacement {report['replacement_score']:.2f}   |   "
        f"error-share {report['error_influence_share']:.2f}",
        fontsize=9,
    )
    plt.tight_layout()
    plt.show()


def serve_viewer(port=8033, stop=False):
    """3b · Serve circuit-tracer's interactive viewer for the shared explore/
    folder (in-process). Run once — it stays up across rebuilds; reopen the new
    ?slug= link after each rebuild. serve_viewer(stop=True) to shut down. Port
    8033 so it does NOT clash with explore_attribution.ipynb (port 8032)."""
    if stop:
        if _LAST.get("server") is not None:
            _LAST["server"].stop()
            _LAST["server"] = None
            print("viewer stopped.")
        return

    if "graph_dir" not in _LAST:
        raise NameError("No graph yet — run feature_graph(prompt) first.")
    features_dir = storage_root() / "clt_features" / SCAN_NAME

    # Reuse one server across rebuilds; stop a stale one first.
    if _LAST.get("server") is not None:
        _LAST["server"].stop()
    _LAST["server"] = start_server(str(_LAST["graph_dir"]), str(features_dir), SCAN_NAME, port=port)

    print(f"Viewer → http://localhost:{port}/?slug={_LAST['slug']}")
    print("Rebuild (re-run section 3) then open the new ?slug= link or use the dropdown.")
    print("serve_viewer(stop=True)  # to shut down")


print("section 3 ready: feature_graph(prompt[, target]), show_top_features([top_k]), serve_viewer([port])")


section 3 ready: feature_graph(prompt[, target]), show_top_features([top_k]), serve_viewer([port])


In [ ]:
GRAPH_TARGET = None   # None -> auto (expected token for `prompt`, else top prediction); or e.g. " August"

feature_graph(prompt, target=GRAPH_TARGET)


### 3a · Inline — top feature nodes

`show_top_features()` ranks feature nodes by their per-node influence on the target logit (circuit-tracer's own `compute_node_influence`) and plots the top `TOP_K`. The full node-link graph is in the web viewer below.

In [ ]:
TOP_K = 15

show_top_features(top_k=TOP_K)


### 3b · Web viewer

`serve_viewer()` serves circuit-tracer's interactive viewer for the shared `explore/` folder (in-process). **Run once** — it stays up across rebuilds. After rebuilding for a new prompt, open the printed `?slug=` link or pick it from the dropdown at the top of the viewer. `serve_viewer(stop=True)` shuts it down.

In [ ]:
serve_viewer()   # run once; stays up across rebuilds.  serve_viewer(stop=True) to shut down.


## 4 · Feature Analysis — which CLT features recall a birth month

For each month, take people born in that month, build the recall prompt (`…was born on`, template 0), run the model, and read the CLT feature activations **at the recall position** (the last token, where the month is predicted). Aggregate across people, then rank features by:

- **`mean_target`** — how hard a feature fires on this month's recall positions, and
- **`specificity = mean_target / mean_other`** — how selectively it fires on this month vs. a shared baseline of all *other* months.

Features are labeled `L{layer} F{idx}` — the same scheme as the Feature Graph (section 3) and the attribution viewer, so a feature you find here can be looked up there.

In [113]:
MONTH_STRINGS = [
    "January", "February", "March",     "April",   "May",      "June",
    "July",    "August",   "September", "October", "November", "December",
]

# In-vocab people grouped by birth month (deterministic dataset order).
people_by_month = defaultdict(list)
for p in sampler.people:
    if name_in_vocab(f" {p['first_name']} {p['last_name']}"):
        people_by_month[p["birthmonth"]].append(p)

def recall_prompt(person, exposure_idx=0):
    """Training-prefix prompt ending right before the birth month (template 0)."""
    bio = sampler.render(person, exposure_idx)
    date = f"{person['birthmonth']} {person['birthday']}, {person['birthyear']}"
    return bio[:bio.index(date)].rstrip()

@torch.no_grad()
def recall_features(prompt):
    """CLT feature activations at the recall position (last token). Returns
    [n_layers, d_transcoder]; position 0 (BOS) is already zeroed by the model."""
    _logits, cache = model.get_activations(prompt)   # [n_layers, seq, d_transcoder]
    return cache[:, -1, :].clone()

def month_feature_stats(month):
    """Per-feature mean for `month` (target) vs all other months (baseline).
    Reads the globals set by the sweep cell."""
    tgt = month_sum[month] / max(month_cnt[month], 1)
    oth = (total_sum - month_sum[month]) / max(total_cnt - month_cnt[month], 1)
    return tgt, oth

def top_features(tgt_mean, other_mean, k=15, by="mean", min_target=0.0):
    """Rank flattened (layer, feature) entries by mean activation or specificity."""
    spec = tgt_mean / other_mean.clamp_min(1e-6)
    if by == "specificity":
        score = spec.clone()
        score[tgt_mean < min_target] = -float("inf")   # floor near-dead features
    else:
        score = tgt_mean
    order = torch.argsort(score.flatten(), descending=True)[:k]
    rows = []
    for f in order.tolist():
        layer, idx = divmod(f, tgt_mean.shape[1])
        rows.append((layer, idx, float(tgt_mean[layer, idx]),
                     float(other_mean[layer, idx]), float(spec[layer, idx])))
    return rows

def print_rows(rows, title):
    print(title)
    print(f"  {'L.feat':>12}  {'mean_tgt':>9}  {'mean_oth':>9}  {'specif':>8}")
    for layer, idx, mt, mo, sp in rows:
        print(f"  {'L'+str(layer)+' F'+str(idx):>12}  {mt:>9.4f}  {mo:>9.4f}  {sp:>8.2f}")

print(f"feature helpers ready · in-vocab people per month: "
      f"{ {m: len(people_by_month[m]) for m in MONTH_STRINGS} }")

feature helpers ready · in-vocab people per month: {'January': 118, 'February': 139, 'March': 132, 'April': 137, 'May': 126, 'June': 127, 'July': 133, 'August': 123, 'September': 125, 'October': 128, 'November': 131, 'December': 132}


### 4a · Sweep — gather recall features per month

Runs `N_PER_MONTH` people through each of the twelve months and accumulates the per-feature activation sum at the recall position. Cached to `clt_storage/clt_feature_explorer/<scan>/` — set `RECOMPUTE = True` (or bump `N_PER_MONTH`) to re-run. ~minutes on CPU the first time; instant once cached.

In [165]:
N_PER_MONTH = 1000    # people sampled per month (CPU inference cost scales with this)
RECOMPUTE   = True  # set True to ignore the cache and re-run the sweep

CACHE_DIR  = storage_root() / "clt_feature_explorer" / SCAN_NAME
CACHE_PATH = CACHE_DIR / f"recall_month_sums_n{N_PER_MONTH}.pt"

if CACHE_PATH.exists() and not RECOMPUTE:
    blob = torch.load(CACHE_PATH, weights_only=False)
    month_sum, month_cnt = blob["sum"], blob["count"]
    print(f"loaded cached sweep from {CACHE_PATH}")
else:
    month_sum, month_cnt = {}, {}
    t0 = time.time()
    for m in MONTH_STRINGS:
        people = people_by_month[m][:N_PER_MONTH]
        s = None
        for p in people:
            feats = recall_features(recall_prompt(p))     # [n_layers, d_transcoder]
            s = feats.clone() if s is None else s + feats
        month_sum[m], month_cnt[m] = s, len(people)
        print(f"  {m:>10}: {len(people):>4} people  ({time.time() - t0:5.1f}s elapsed)")
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    torch.save({"sum": month_sum, "count": month_cnt}, CACHE_PATH)
    print(f"wrote {CACHE_PATH}")

total_sum = sum(month_sum[m] for m in MONTH_STRINGS)
total_cnt = sum(month_cnt[m] for m in MONTH_STRINGS)
print(f"\ntotal recall positions swept: {total_cnt:,}  (feature tensor {tuple(total_sum.shape)})")

     January:  118 people  (  0.8s elapsed)
    February:  139 people  (  1.8s elapsed)
       March:  132 people  (  2.7s elapsed)
       April:  137 people  (  3.8s elapsed)
         May:  126 people  (  4.7s elapsed)
        June:  127 people  (  5.7s elapsed)
        July:  133 people  (  6.7s elapsed)
      August:  123 people  (  7.7s elapsed)
   September:  125 people  (  8.6s elapsed)
     October:  128 people  (  9.5s elapsed)
    November:  131 people  ( 10.4s elapsed)
    December:  132 people  ( 11.4s elapsed)
wrote /Users/efmac/Code/Project Code/CRL-Interp/Interp_LM4/clt_storage/clt_feature_explorer/grid-L4-H6/recall_month_sums_n1000.pt

total recall positions swept: 1,551  (feature tensor (4, 6144))


### 4b · Inspect one month

Pick a `MONTH` and rank its recall features by raw **mean activation** and by **specificity** (`mean_target / mean_other`). `MIN_TARGET` floors near-dead features before the specificity ranking.

In [154]:
MONTH = "October"     # any month from MONTH_STRINGS
K = 15
MIN_TARGET = 0.005   # floor mean_target before ranking by specificity

tgt_mean, other_mean = month_feature_stats(MONTH)
print(f"=== {MONTH}  (n={month_cnt[MONTH]} people, "
      f"baseline {total_cnt - month_cnt[MONTH]:,} other-month positions) ===\n")
print_rows(top_features(tgt_mean, other_mean, k=K, by="mean"),
           f"Top {K} by mean activation:")
print()
print_rows(top_features(tgt_mean, other_mean, k=K, by="specificity", min_target=MIN_TARGET),
           f"Top {K} by specificity (mean_target >= {MIN_TARGET}):")

=== October  (n=128 people, baseline 1,423 other-month positions) ===

Top 15 by mean activation:
        L.feat   mean_tgt   mean_oth    specif
      L3 F4380    90.8188     0.2059    441.02
      L3 F2427    88.4498    50.4858      1.75
      L3 F4183    75.9700    45.1856      1.68
      L2 F5546    59.9478    60.2754      0.99
      L1 F4580    54.8935    55.3333      0.99
      L1 F1455    54.1898    53.9359      1.00
      L3 F3910    53.0836    38.7973      1.37
      L2 F3606    50.5653    50.9564      0.99
      L2 F1070    49.9355    49.3769      1.01
      L2 F2197    46.4786    47.0266      0.99
      L2 F2894    43.4440    43.1707      1.01
      L1 F4009    42.1616    42.3866      0.99
       L2 F578    36.7273    36.4553      1.01
      L2 F2602    34.5405    34.9754      0.99
      L1 F1335    33.6225    33.8959      0.99

Top 15 by specificity (mean_target >= 0.005):
        L.feat   mean_tgt   mean_oth    specif
      L3 F3367     7.0481     0.0000  7048064.00
      L

### 4c · All months at a glance

Top features by specificity, one block per month. The `other` baseline is shared (all positions not in that month), so the specificity numbers are directly comparable across the twelve breakdowns.

In [151]:
TOP_K = 20
MIN_TARGET = 0.005

for m in MONTH_STRINGS:
    if month_cnt.get(m, 0) == 0:
        print(f"\n=== {m}: no people ===")
        continue
    tgt_mean, other_mean = month_feature_stats(m)
    rows = top_features(tgt_mean, other_mean, k=TOP_K, by="mean", min_target=MIN_TARGET)
    print(f"\n=== {m}  (n={month_cnt[m]}) — top {TOP_K} by mean ===")
    print(f"  {'L.feat':>12}  {'mean_tgt':>9}  {'mean_oth':>9}  {'specif':>8}")
    for layer, idx, mt, mo, sp in rows:
        print(f"  {'L'+str(layer)+' F'+str(idx):>12}  {mt:>9.4f}  {mo:>9.4f}  {sp:>8.2f}")


=== January  (n=118) — top 20 by mean ===
        L.feat   mean_tgt   mean_oth    specif
      L3 F5934   100.8192     0.1674    602.37
      L3 F1975    91.2343     0.1242    734.75
      L2 F5546    59.9052    60.2766      0.99
      L1 F4580    54.9903    55.3222      0.99
      L1 F1455    54.0655    53.9479      1.00
      L3 F4183    52.4930    47.3336      1.11
      L2 F3606    50.6365    50.9479      0.99
      L2 F1070    49.7674    49.3946      1.01
      L3 F1768    48.0498    36.5491      1.31
      L2 F2197    46.8860    46.9892      1.00
      L3 F3013    44.5645     0.4254    104.75
      L2 F2894    43.0999    43.2009      1.00
      L1 F4009    42.3319    42.3710      1.00
       L2 F578    36.0215    36.5153      0.99
      L2 F2602    34.2973    34.9924      0.98
      L3 F3910    33.8317    40.4823      0.84
      L1 F1335    33.3571    33.9158      0.98
      L3 F2427    31.6918    55.4244      0.57
       L1 F972    29.6504    29.8436      0.99
      L1 F5664   